# Library

In [11]:
# import packages
import numpy as np
np.random.seed(4)
import matplotlib.pyplot as plt

from itertools import groupby
import pandas as pd
import datetime as dt

import os
import re
import string

import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm as tq
tq.pandas() #thanks to https://stackoverflow.com/questions/18603270/progress-indicator-during-pandas-operations


# Investigate new dataset + import it

## Briefly investigate

In [34]:
# import and show the 3 sheets: RGDPQ
sheets = ['RGDPQ', 'UnempQ','UnempM']
dfs = pd.read_excel('CountryVariables/MissingExtra/addOnMissing.xlsx', sheet_name=sheets)

# Access dataframes by sheet name
dfgdpQ = dfs[sheets[0]]
dfunempQ = dfs[sheets[1]]
dfunempM = dfs[sheets[2]]


In [38]:
for i in range(len(sheets)):
    print('Considering sheets '+ sheets[i])
    print(dfs[sheets[i]].columns.tolist())
    print('Size of '+ str(dfs[sheets[i]].shape))
    print('Countries available: ')
    print(dfs[sheets[i]]['country'].value_counts())
    print('\n')


Considering sheets RGDPQ
['date', 'RGDP', 'gRGDP', 'country']
Size of (568, 4)
Countries available: 
country
Chile          100
NewZealand     100
Philippines    100
Thailand       100
Peru            89
Colombia        79
Name: count, dtype: int64


Considering sheets UnempQ
['date', 'UnemployRateTotal', 'country']
Size of (212, 3)
Countries available: 
country
NewZealand     100
Philippines     64
Thailand        48
Name: count, dtype: int64


Considering sheets UnempM
['date', 'UnemployRateTotal', 'country']
Size of (335, 3)
Countries available: 
country
Peru           286
Philippines     49
Name: count, dtype: int64




## Merge with the main dataset
with CBMonthly and CBQuarterly in particular

### Quarterly


In [13]:
# import and show the 3 sheets: RGDPQ
sheets = ['RGDPQ', 'UnempQ','UnempM']
dfs = pd.read_excel('CountryVariables/MissingExtra/addOnMissing.xlsx', sheet_name=sheets)

# Access dataframes by sheet name
dfgdpQ = dfs[sheets[0]]
dfunempQ = dfs[sheets[1]]
dfunempM = dfs[sheets[2]]

dfQ = pd.read_csv('CBQuarterly.csv', parse_dates = ['dateq'])
dfQ['dateq'] = pd.PeriodIndex(dfQ.dateq, freq='Q')
print(dfQ.shape)
print(dfQ.columns.tolist())


(1592, 105)
['country', 'dateq', 'numWords', 'numSentence', 'numWordspSentence', 'policyRate', 'positiveFin', 'neutralFin', 'negativeFin', 'score_hawkish_wNeg', 'Financial Sector', 'Central Bank', 'Government', 'Households', 'Firms', 'positiveT', 'negativeT', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'contains_lorem ipsum', 'duplicate_line_chr_fraction', 'duplicate_paragraph_chr_fraction', 'duplicate_ngram_chr_fraction_5', 'duplicate_ng

In [15]:
# RGDP to quarterly format
dfgdpQ['dateq'] = pd.PeriodIndex(dfgdpQ.date, freq='Q')
dfgdpQ.drop(columns = ['date'], inplace =True)
print(dfgdpQ.head())

# Unemployment to quarterly
dfunempQ['dateq'] = pd.PeriodIndex(dfunempQ.date, freq='Q')
dfunempQ.drop(columns = ['date'], inplace =True)
print(dfunempQ.head())
dfunempM['dateq'] = pd.PeriodIndex(dfunempM.date, freq='Q')
dfunempM.drop(columns = ['date'], inplace =True)
print(dfunempM.head())

# need to average it up by date
dfunempM = dfunempM.groupby(['country','dateq']).mean().reset_index()
print(dfunempM.head())

# concat to the one already quarterly
dfunemp = pd.concat([dfunempQ, dfunempM])

# sort and reset index
dfunemp = dfunemp.sort_values(by = ['country','dateq'])
print(dfunemp.head())


   RGDP     gRGDP country   dateq
0   NaN  0.944780   Chile  2000Q1
1   NaN  0.221694   Chile  2000Q2
2   NaN  1.555900   Chile  2000Q3
3   NaN  1.173626   Chile  2000Q4
4   NaN  0.696727   Chile  2001Q1
   UnemployRateTotal     country   dateq
0                6.5  NewZealand  2000Q1
1                6.3  NewZealand  2000Q2
2                6.0  NewZealand  2000Q3
3                5.8  NewZealand  2000Q4
4                5.5  NewZealand  2001Q1
   UnemployRateTotal country   dateq
0           9.240754    Peru  2001Q2
1           9.489724    Peru  2001Q2
2           9.196063    Peru  2001Q3
3           9.491484    Peru  2001Q3
4           9.548051    Peru  2001Q3
  country   dateq  UnemployRateTotal
0    Peru  2001Q2           9.365239
1    Peru  2001Q3           9.411866
2    Peru  2001Q4           9.342992
3    Peru  2002Q1           9.972273
4    Peru  2002Q2          10.250034
   UnemployRateTotal     country   dateq
0                6.5  NewZealand  2000Q1
1                6.3  Ne

In [17]:
# before adding in checking which is missing for both RGDP and unemployment rate
dfFinalQ = dfQ.copy()
print(dfFinalQ.shape)
dfFinalQ = pd.merge(dfFinalQ, dfgdpQ, on = ['country','dateq'], how = 'left', suffixes = ['','x'])
dfFinalQ = pd.merge(dfFinalQ, dfunemp, on = ['country','dateq'], how = 'left', suffixes = ['','y'])
print(dfFinalQ.shape)
for i in ['RGDP','UnemployRateTotal']:
    print('For '+i)
    print(dfFinalQ[dfFinalQ[i].isnull()]['country'].value_counts())
    print('\n')

for i in ['RGDPx','gRGDPx','UnemployRateTotaly']:
    print('For '+i)
    print(dfFinalQ[~dfFinalQ[i].isnull()]['country'].value_counts())
    print('\n')

# now fill in the blank for both RGDP and unemployment rate
dfFinalQ['RGDP'] = dfFinalQ['RGDP'].fillna(dfFinalQ['RGDPx'])
dfFinalQ['gRGDP'] = dfFinalQ['gRGDP'].fillna(dfFinalQ['gRGDPx'])
dfFinalQ['UnemployRateTotal'] = dfFinalQ['UnemployRateTotal'].fillna(dfFinalQ['UnemployRateTotaly'])

# Check if the amount filled in is ri['RGDP','gRGDP','RGDPx','gRGDPx','UnemployRateTotal','UnemployRateTotaly','country']ght
listToCheck = ['RGDP','gRGDP','UnemployRateTotal']
for i in listToCheck:
    print('For '+i)
    print(dfFinalQ[dfFinalQ[i].isnull()]['country'].value_counts())
    print('\n')
                         


(1592, 105)
(1592, 108)
For RGDP
country
NewZealand     100
Chile           99
Peru            96
Philippines     93
Thailand        92
Armenia         64
Colombia        38
Norway           2
Indonesia        2
Fed              1
Hungary          1
Iceland          1
Australia        1
Japan            1
Korea            1
ECB              1
Canada           1
Poland           1
Name: count, dtype: int64


For UnemployRateTotal
country
NewZealand     100
Peru            96
Philippines     93
Thailand        92
Indonesia       77
Armenia         64
Iceland          1
Name: count, dtype: int64


For RGDPx
country
NewZealand     100
Philippines     93
Thailand        92
Peru            88
Colombia        37
Name: count, dtype: int64


For gRGDPx
country
NewZealand     100
Chile           99
Philippines     93
Thailand        92
Peru            87
Colombia        37
Name: count, dtype: int64


For UnemployRateTotaly
country
NewZealand     100
Peru            95
Philippines     80
Thailand

In [19]:
# drop the two extra columns
dfFinalQ = dfFinalQ.drop(columns = ['RGDPx','gRGDPx','UnemployRateTotaly'], errors = 'ignore')
print(dfFinalQ.shape)


(1592, 105)


In [21]:
dfFinalQ.to_csv('CBQuarterly.csv', index = False)


### Monthly

In [23]:
# import and show the 3 sheets: RGDPQ
sheets = ['RGDPQ', 'UnempQ','UnempM']
dfs = pd.read_excel('CountryVariables/MissingExtra/addOnMissing.xlsx', sheet_name=sheets)

# Access dataframes by sheet name
dfgdpQ = dfs[sheets[0]]
dfunempQ = dfs[sheets[1]]
dfunempM = dfs[sheets[2]]

dfM = pd.read_csv('CBMonthly.csv', parse_dates = ['dateym'])
print(dfM.shape)
print(dfM.columns.tolist())


(3702, 105)
['country', 'dateym', 'numWords', 'numSentence', 'numWordspSentence', 'policyRate', 'positiveFin', 'neutralFin', 'negativeFin', 'score_hawkish_wNeg', 'Financial Sector', 'Central Bank', 'Government', 'Households', 'Firms', 'positiveT', 'negativeT', 'concreteness', 'concretenessExtra', 'concreteRatio', 'concreteMean', 'concrete25', 'concreteMeanExtra', 'concrete25Extra', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences', 'passed_quality_check', 'n_stop_words', 'alpha_ratio', 'mean_word_length', 'doc_length', 'symbol_to_word_ratio_#', 'proportion_ellipsis', 'proportion_bullet_points', 'contains_lorem ipsum', 'duplicate_line_chr_fraction', 'duplicate_paragraph_chr_fraction', 'duplicate_ngram_chr_fraction_5', 'duplicate_n

In [25]:
# RGDP to quarterly format
dfFinalM = dfM.copy()
dfFinalM['dateq'] = pd.PeriodIndex(dfFinalM.dateym, freq='Q')
dfFinalM['dateym2'] = dfFinalM['dateym'].dt.to_period('M')
print(dfFinalM.shape)

# Unemployment to quarterly
dfgdpQ['dateq'] = pd.PeriodIndex(dfgdpQ.date, freq='Q')
dfgdpQ.drop(columns = ['date'], inplace =True)
dfunempQ['dateq'] = pd.PeriodIndex(dfunempQ.date, freq='Q')
dfunempQ.drop(columns = ['date'], inplace =True)

# concat to the one already quarterly
dfunempM['dateym'] = dfunempM['date'].dt.to_period('M')
dfunempM.drop(columns = ['date'], inplace =True)
print(dfunempM.head())


(3702, 107)
   UnemployRateTotal country   dateym
0           9.240754    Peru  2001-05
1           9.489724    Peru  2001-06
2           9.196063    Peru  2001-07
3           9.491484    Peru  2001-08
4           9.548051    Peru  2001-09


In [27]:
# now fill in the blank for both RGDP and unemployment rate
# before adding in checking which is missing for both RGDP and unemployment rate
dfFinalM = pd.merge(dfFinalM, dfgdpQ, on = ['country','dateq'], how = 'left', suffixes = ['','x'])
dfFinalM = pd.merge(dfFinalM, dfunempQ, on = ['country','dateq'], how = 'left', suffixes = ['','y'])
dfFinalM = pd.merge(dfFinalM, dfunempM, left_on = ['country','dateym2'], right_on = ['country','dateym'], how = 'left', suffixes = ['','z'])
print(dfFinalM.shape)
for i in ['RGDP','UnemployRateTotal']:
    print('For '+i)
    print(dfFinalM[dfFinalM[i].isnull()]['country'].value_counts())
    print('\n')

for i in ['RGDPx','gRGDPx','UnemployRateTotaly','UnemployRateTotalz']:
    print('For '+i)
    print(dfFinalM[~dfFinalM[i].isnull()]['country'].value_counts())
    print('\n')

# now fill in the blank for both RGDP and unemployment rate
dfFinalM['RGDP'] = dfFinalM['RGDP'].fillna(dfFinalM['RGDPx'])
dfFinalM['gRGDP'] = dfFinalM['gRGDP'].fillna(dfFinalM['gRGDPx'])
dfFinalM['UnemployRateTotal'] = dfFinalM['UnemployRateTotal'].fillna(dfFinalM['UnemployRateTotaly'])
dfFinalM['UnemployRateTotal'] = dfFinalM['UnemployRateTotal'].fillna(dfFinalM['UnemployRateTotalz'])

# Check if the amount filled in is ri['RGDP','gRGDP','RGDPx','gRGDPx','UnemployRateTotal','UnemployRateTotaly','country']ght
listToCheck = ['RGDP','gRGDP','UnemployRateTotal']
for i in listToCheck:
    print('For '+i)
    print(dfFinalM[dfFinalM[i].isnull()]['country'].value_counts())
    print('\n')
                         


(3702, 112)
For RGDP
country
Peru           287
Chile          261
Philippines    207
Thailand       175
Armenia        143
NewZealand     100
Colombia        87
Indonesia        5
Norway           4
Poland           3
ECB              2
Hungary          2
Iceland          2
Korea            2
Fed              1
Australia        1
Japan            1
Canada           1
Name: count, dtype: int64


For UnemployRateTotal
country
Peru           287
Indonesia      212
Philippines    207
Thailand       175
Armenia        143
NewZealand     100
Iceland          2
ECB              1
Norway           1
Poland           1
Name: count, dtype: int64


For RGDPx
country
Peru           264
Philippines    207
Thailand       175
NewZealand     100
Colombia        86
Name: count, dtype: int64


For gRGDPx
country
Chile          261
Peru           261
Philippines    207
Thailand       175
NewZealand     100
Colombia        86
Name: count, dtype: int64


For UnemployRateTotaly
country
Philippines    138
N

In [29]:
# drop the two extra columns
dfFinalM = dfFinalM.drop(columns = ['dateq', 'dateym2', 'RGDPx', 'gRGDPx', 'UnemployRateTotaly', 'UnemployRateTotalz', 'dateymz'], errors = 'ignore')
print(dfFinalM.shape)


(3702, 105)


In [31]:
dfFinalM.to_csv('CBMonthly.csv', date_format='%Y-%m-%d', index = False)
